# **Tutorial 05 - Penalized Least Squares**: Supplementary Interactive Material

-------------------------------------------------------------------------

A least squares fit can follow noisy observations too closely. We explore how a penalty changes the fit, then use a related idea to fill missing image pixels.

In [ ]:
from pathlib import Path
from io import BytesIO
import base64
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from PIL import Image


def replace_plot(fig, output):
    # Replace the output directly; do not leave figures for automatic display.
    try:
        with BytesIO() as buffer:
            fig.savefig(buffer, format='png', bbox_inches='tight')
            png = base64.b64encode(buffer.getvalue()).decode('ascii')
    finally:
        plt.close(fig)
    output.outputs = ({'output_type': 'display_data',
                       'data': {'image/png': png}, 'metadata': {}},)

## Example 01: Ridge regression

We fit $p(t)=x_0+x_1t+\cdots+x_{12}t^{12}$ to 18 noisy samples of $\sin(\pi t)$. Each row of $A$ is $[1,t_i,\ldots,t_i^{12}]$, and $b$ contains the observations. Ridge chooses the 13 coefficients by minimizing

$$\frac12\|Ax-b\|_2^2+\frac{\lambda}{2}\|x\|_2^2.$$

The first term measures fitting error; the second penalizes large coefficients. With a small penalty, the polynomial can wiggle to follow noise. Increasing $\lambda$ shrinks its coefficients; too large a penalty flattens the curve. Here the intercept is penalized too.

**Move the slider: when do the wiggles disappear, and when does the fit become too flat?**

In [ ]:
# Close the previous controls if this cell is rerun.
if 'ridge_panel' in globals():
    ridge_panel.close()
    ridge_slider.close()
    ridge_output.close()

rng = np.random.default_rng(12)
t = np.linspace(-1, 1, 18)
b = np.sin(np.pi * t) + 0.15 * rng.standard_normal(t.size)
grid = np.linspace(-1, 1, 400)
A = np.vander(t, N=13, increasing=True)
A_grid = np.vander(grid, N=13, increasing=True)
ridge_slider = widgets.FloatLogSlider(value=0.01, base=10, min=-7, max=2,
                                      step=0.25, description='Penalty', continuous_update=False)
ridge_output = widgets.Output()


def plot_ridge(change=None):
    lam = ridge_slider.value
    augmented_A = np.vstack([A, np.sqrt(lam) * np.eye(A.shape[1])])
    augmented_b = np.concatenate([b, np.zeros(A.shape[1])])
    x = np.linalg.lstsq(augmented_A, augmented_b, rcond=None)[0]
    with plt.ioff():
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.scatter(t, b, color='black', label='Observations')
        ax.plot(grid, A_grid @ x, label='Ridge fit')
        ax.set(xlabel='t', ylabel='Function value', title=f'Ridge regression: lambda = {lam:g}')
        ax.legend()
        replace_plot(fig, ridge_output)


ridge_slider.observe(plot_ridge, names='value')
ridge_panel = widgets.VBox([ridge_slider, ridge_output])
display(ridge_panel)
plot_ridge()

## Example 02: Revisiting the p norm

Recall the unit ball from Tutorial 01:

$$B_p=\{(x,y):\|(x,y)\|_p\leq1\},\qquad
\|(x,y)\|_p=(|x|^p+|y|^p)^{1/p}.$$

Move the slider from $p=0.01$ to $100$, or select Infinity. At $p=1$ the ball is a diamond, at $p=2$ a circle, and as $p\to\infty$ it approaches the square $\max(|x|,|y|)\leq1$. For $0<p<1$, this is a quasi-norm rather than a norm. Near zero, the ball becomes very thin along the coordinate axes; the formula is undefined at $p=0$.

**Compare $p=2,10,100$, and infinity. Which points stay on the boundary?**

In [ ]:
if 'p_panel' in globals():
    p_panel.close()
    p_choice.close()
    p_output.close()
if 'p_infinity' in globals():
    p_infinity.close()

p_choice = widgets.FloatLogSlider(value=2.0, base=10, min=-2, max=2,
                                  step=0.02, description='p:', readout_format='.2f',
                                  continuous_update=False)
p_infinity = widgets.Checkbox(value=False, description='Infinity')
p_output = widgets.Output()


def plot_p_norm(change=None):
    p_choice.disabled = p_infinity.value
    p = np.inf if p_infinity.value else p_choice.value
    theta = np.linspace(0, 2 * np.pi, 2001)
    directions = np.column_stack([np.cos(theta), np.sin(theta)])
    if np.isinf(p):
        boundary = directions / np.max(np.abs(directions), axis=1)[:, None]
    else:
        # Logarithms avoid overflow in the scale when p is near zero.
        directions[np.abs(directions) < 1e-14] = 0.0
        with np.errstate(divide='ignore'):
            log_abs = np.log(np.abs(directions))
        log_scale = np.logaddexp(p * log_abs[:, 0], p * log_abs[:, 1]) / p
        boundary = np.sign(directions) * np.exp(log_abs - log_scale[:, None])
    label = r'\infty' if np.isinf(p) else f'{p:g}'
    with plt.ioff():
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.fill(boundary[:, 0], boundary[:, 1], alpha=0.3)
        ax.plot(boundary[:, 0], boundary[:, 1])
        ax.set(xlim=(-1.3, 1.3), ylim=(-1.3, 1.3), xlabel='x', ylabel='y',
               title=rf'Unit ball: $p={label}$')
        ax.set_aspect('equal')
        replace_plot(fig, p_output)


p_choice.observe(plot_p_norm, names='value')
p_infinity.observe(plot_p_norm, names='value')
p_panel = widgets.VBox([p_choice, p_infinity, p_output])
display(p_panel)
plot_p_norm()

## Example 03: Image recovery with total variation

Let $f$ be the observed image and $u$ the recovered image. We balance:

1. **Data fidelity:** Resemble the observations.
2. **Regularity:** Avoid unnecessary fluctuations.

$$
\boxed{
\min_u
\underbrace{\frac12\int_\Omega (u-f)^2\,dx\,dy}_{\text{Data fidelity}}
+\lambda
\underbrace{\int_\Omega |\nabla u|\,dx\,dy}_{\text{Total variation}}
}
$$

- $\Omega$: image region; $\nabla u=(u_x,u_y)$: horizontal and vertical brightness changes.
- TV sums their magnitude $|\nabla u|=\sqrt{u_x^2+u_y^2}$ across the image, discouraging fluctuations while allowing edges.
- Small $\lambda$: closer to observations, including noise. Large $\lambda$: stronger regularity, potentially losing texture.

### Our example: the discrete problem

- Remove 70% of pixels. Let $K$ contain the known pixel locations; black placeholders are not observations.
- Approximate derivatives locally using forward differences (unit pixel spacing):

$$ (D_xu)_{i,j}=u_{i,j+1}-u_{i,j},\qquad (D_yu)_{i,j}=u_{i+1,j}-u_{i,j}. $$

Set the outward differences to zero at the right and bottom boundaries. The code solves

$$
\boxed{
\min_{0\leq u\leq1}\ E(u)=
\underbrace{\frac12\sum_{(i,j)\in K}(u_{i,j}-f_{i,j})^2}_{\text{Data fidelity}}
+\lambda\underbrace{\sum_{i,j}
\left[\sqrt{(D_xu)_{i,j}^2+(D_yu)_{i,j}^2+\varepsilon^2}-\varepsilon\right]}_{\text{Smoothed TV}}
}
$$

- $\varepsilon=0.01$ makes TV differentiable; subtracting $\varepsilon$ gives a constant image TV zero.
- Fidelity uses only known pixels. Their values may change as the solver balances fidelity against TV.

### Gradient descent

Start with the observed values and fill missing pixels with the observed mean. Repeatedly take a step that reduces the **combined objective**:

$$u^{k+1}=\operatorname{clip}_{[0,1]}\!\left(u^k-\alpha_k\nabla E(u^k)\right).$$

Backtracking reduces the step size if necessary. Convexity makes every local minimum global; iterations are still needed to find one.

**Change $\lambda$: which features survive, and which textures are lost?**

In [ ]:
# Locate the same image from either the repository root or Tutorials/.
image_path = next((p for p in [Path('../Data/Beltrami.jpg'), Path('Data/Beltrami.jpg')]
                   if p.is_file()), None)
if image_path is None:
    raise FileNotFoundError('Open this notebook from the repository root or Tutorials/; Data/Beltrami.jpg is required.')
with Image.open(image_path) as source_image:
    small_image = source_image.convert('L')
    small_image.thumbnail((128, 128))  # Keep the computation quick.
    img = np.asarray(small_image, dtype=np.float64) / 255.0


def tv_energy_gradient(u, epsilon=0.01):
    dx = np.zeros_like(u)
    dy = np.zeros_like(u)
    dx[:, :-1] = u[:, 1:] - u[:, :-1]
    dy[:-1, :] = u[1:, :] - u[:-1, :]
    magnitude = np.sqrt(dx**2 + dy**2 + epsilon**2)
    px, py = dx / magnitude, dy / magnitude
    gradient = np.zeros_like(u)
    gradient[:, :-1] -= px[:, :-1]
    gradient[:, 1:] += px[:, :-1]
    gradient[:-1, :] -= py[:-1, :]
    gradient[1:, :] += py[:-1, :]
    return np.sum(magnitude - epsilon), gradient


def recovery_energy_gradient(u, observed, known, lam, epsilon=0.01):
    tv, tv_gradient = tv_energy_gradient(u, epsilon)
    residual = np.where(known, u - observed, 0.0)
    energy = 0.5 * np.sum(residual**2) + lam * tv
    gradient = residual + lam * tv_gradient
    return energy, gradient


def tv_inpaint(observed, known, initial, lam=0.02, n_iter=150, epsilon=0.01):
    # Ground truth at missing pixels is never passed to the solver.
    if lam < 0:
        raise ValueError('The penalty weight must be nonnegative.')
    u = np.clip(initial.copy(), 0, 1)
    energy, gradient = recovery_energy_gradient(u, observed, known, lam, epsilon)
    history = [energy]
    step = 0.2
    for _ in range(n_iter):
        step = min(1.2 * step, 1.0)
        for _ in range(30):
            candidate = np.clip(u - step * gradient, 0, 1)
            change_squared = np.sum((candidate - u)**2)
            new_energy, new_gradient = recovery_energy_gradient(candidate, observed, known, lam, epsilon)
            if new_energy <= energy - 1e-4 * change_squared / step:
                break
            step *= 0.5
        else:
            raise RuntimeError('Backtracking failed to find a descent step.')
        u, energy, gradient = candidate, new_energy, new_gradient
        history.append(energy)
        if np.sqrt(change_squared / u.size) < 1e-6:
            break
    return u, np.asarray(history)

In [ ]:
missing_fraction = 0.7  # Try 0.3 or 0.9.
lam = 0.02            # Try 0.002, 0.02, or 0.2.
iterations = 150       # Try 0, 50, or 500.
known = np.random.default_rng(42).uniform(size=img.shape) > missing_fraction
observed = np.where(known, img, 0.0)
initial = np.where(known, observed, img[known].mean())
recovered, history = tv_inpaint(observed, known, initial, lam=lam, n_iter=iterations)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image, title in zip(
    axes, [img, observed, recovered],
    ['Original', f'{(~known).mean():.0%} missing', f'TV reconstruction (lambda={lam:g})']
):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

mse = np.mean((recovered[~known] - img[~known])**2)
print(f'Missing-pixel MSE: {mse:.4f}')

> **Explore:** Vary $\lambda$, the missing fraction, and the iteration count. What happens to edges and texture?
>
> The solver reduces fidelity error plus weighted TV; a lower objective need not mean a more accurate image. The original image is used only to evaluate error at missing pixels.